In [0]:
-- 1. Create a staging view to clean data and cast dates
CREATE STREAMING LIVE VIEW stg_customers AS
SELECT
  CAST(customer_id AS INT) AS customer_id,
  CAST(customer_acct_num AS BIGINT) AS customer_acct_num,
  first_name,
  last_name,
  
  -- Use M/d and yyyy-M-d to handle both single and double-digit months/days
  COALESCE(
    try_to_date(acct_open_date, 'M/d/yyyy'),
    try_to_date(acct_open_date, 'yyyy-M-d'),
    try_to_date(acct_open_date, 'd-MMM-yy'),
    CAST(acct_open_date AS DATE) -- Fallback just in case Spark natively parses it
  ) AS acct_open_date,
  
  COALESCE(
    try_to_date(birthdate, 'M/d/yyyy'),
    try_to_date(birthdate, 'yyyy-M-d'),
    try_to_date(birthdate, 'd-MMM-yy'),
    CAST(birthdate AS DATE)
  ) AS birthdate,
  
  customer_address,
  customer_city,
  customer_state_province,
  customer_country,
  CAST(customer_postal_code AS STRING) AS customer_postal_code,
  
  -- Clean up string anomalies (e.g., standardizing casing)
  UPPER(gender) AS gender,
  UPPER(marital_status) AS marital_status,
  
  education,
  occupation,
  homeowner,
  yearly_income,
  
  CAST(num_children_at_home AS INT) AS num_children_at_home,
  CAST(total_children AS INT) AS total_children,
  member_card,
  
  _ingestion_timestamp
FROM STREAM(mavenmarket.bronze.customers)
-- Handle critical anomaly: drop records with no identifiable customer key
WHERE customer_id IS NOT NULL;


-- 2. Define the target table for SCD Type 2
CREATE STREAMING LIVE TABLE silver_customers
COMMENT "Cleaned customer dimension with SCD Type 2 history";


-- 3. Apply changes to execute the SCD Type 2 logic
APPLY CHANGES INTO live.silver_customers
FROM STREAM(live.stg_customers)
  KEYS (customer_id)
  SEQUENCE BY _ingestion_timestamp
  STORED AS SCD TYPE 2;